# Step 1b · Data and panel (2020–2021 events, visits instead of trips)

**Why 1b**: Step 1 found that BTS trip counts do not pick up the effect of outages. BTS counts how many trips a county's residents make, wherever they are, against a fixed resident population; evacuating, cleaning up and looking for supplies all count as trips, so trips in hard-hit counties sometimes go up. GeoDS visits count how often the shops and places in a county are visited, and these drop clearly in hard-hit counties. Step 1b therefore makes GeoDS visits the main activity measure. GeoDS and Womply both end in 2021, so the events change to 8 events from 2020–2021.

**Events**: hurricanes Isaias, Laura, Delta, Zeta and Ida, the August 2020 Midwest derecho (a long-lived windstorm), the October 2020 Oklahoma ice storm, and the February 2021 winter storm Uri.

**The main result is a weekly panel**: "event × county × week", 12 weeks per event (weeks −5 to 6); week 0 is the Monday–Sunday week that contains day 0.

| Data | Content | Placed by | Frequency |
| --- | --- | --- | --- |
| EAGLE-I | Customers without power | County | Every 15 minutes, summed to days and weeks |
| GeoDS (main) | Visits to places, county to county | County of the place visited / county where visitors live | Weekly |
| Womply | Small-business revenue, small businesses open | County where the business is | Weekly |
| Affinity | Card spending | County where the cardholder lives | Daily (7-day moving average); the Sunday value is used for each week |
| BTS (secondary) | Residents' trips, share leaving home | County where residents live | Daily |
| GeoDS 2019 | County-to-county visits over 8 weeks in September–October, the pre-pandemic dependency network | | Summed |

**Structure**

| Part | Content | Approx. time |
| --- | --- | --- |
| A | Events and study areas | 1 minute |
| B | Counties, population, customers | 1 minute |
| C | EAGLE-I outages (the 2020 and 2021 files, about 2.3 GB in all) | 10–20 minutes |
| D | BTS travel (secondary) | 3–8 minutes |
| E | GeoDS visits (56 weeks, about 4 GB in all) | 10–25 minutes |
| F | Business and spending | 1 minute |
| G | Merge into the daily and weekly panels | 2 minutes |
| H | Coverage, three figures, checklist | 1 minute |

**Running the notebook**

- No Earth Engine is needed, only Google Drive. Results go to the `outage_mobility` folder in Drive, with file names starting `step1b_`, so Step 1's results are not overwritten
- Cells run in order from top to bottom. Downloads and intermediate results are saved to Drive as soon as they are done, so a rerun after an interruption skips what is already finished
- Each GeoDS week is saved, filtered, in the `outage_mobility/geods_weeks` folder; later steps use these files too

In [ ]:
# 1. Environment, folders, chart style
import os, io, time
from urllib.parse import quote
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, NullFormatter, LogLocator
from IPython.display import display

try:                                    # in Colab: mount Google Drive
    from google.colab import drive
    drive.mount("/content/drive")
    BASE, TMP = "/content/drive/MyDrive", "/content/tmp"
except ImportError:                     # in local Jupyter: use the current folder
    BASE, TMP = ".", "tmp"
OUT = f"{BASE}/outage_mobility"         # results of this project
P2  = f"{BASE}/us_flood_outage"         # the companion project's folder (may be absent)
GEO_CACHE = f"{OUT}/geods_weeks"        # one GeoDS file per week
for d in (OUT, TMP, GEO_CACHE):
    os.makedirs(d, exist_ok=True)
print("Output folder:", OUT, "| companion project folder found:", os.path.isdir(P2))

def tidy(df, n=1):
    # round numeric columns only, to keep tables readable
    return df.round({c: n for c in df.select_dtypes("number").columns})

def spearman(a, b):
    # Spearman rank correlation: rank first, then take the ordinary correlation
    ok = a.notna() & b.notna()
    return a[ok].rank().corr(b[ok].rank())

def as_day(s):
    # one date type (day, nanosecond unit) so dates from different sources can be merged
    return pd.to_datetime(s).dt.normalize().astype("datetime64[ns]")

def monday(s):
    # the Monday of the week each date falls in
    s = as_day(s)
    return s - pd.to_timedelta(s.dt.dayofweek, unit="D")

# Chart style (shared by all figures). Outage-duration groups use one blue from light to dark; "no substantial outage" is a gray dashed line
INK, MUTED, GRID, SURFACE, FAINT = "#0b0b0b", "#52514e", "#e6e5e1", "#fcfcfb", "#b9b8b2"
C_POINT = "#2a78d6"
WEEK_TICKS = [-4, -2, 0, 2, 4, 6]
GROUPS = [("none", "No outage of 10%+", "#7a7974", "--"),
          ("lt24", "Under 24 h",        "#86b6ef", "-"),
          ("24to72", "24–72 h",         "#2a78d6", "-"),
          ("ge72", "72 h or more",      "#104281", "-")]
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
    "text.color": INK, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False, "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
})

## Parameters

After changing a parameter, rerun from the first part that uses it. Files already downloaded are not downloaded again.

One change from Step 1: **outage days or weeks in the baseline no longer remove the whole county**; only those days or weeks are left out of the baseline. In Step 1, 96% of the flagged counties had just 1–3 days of brief outage, yet they were removed, along with several of Laura's hardest-hit counties.

In [ ]:
# 2. Parameters
WEEK_FROM, WEEK_TO = -5, 6         # each event spans weeks −5 to 6, 12 weeks in all
BASE_WEEKS = (-5, -2)              # weekly baseline: weeks −5 to −2; week −1 holds preparation and evacuation, so it is left out
MIN_BASE_WEEKS = 2                 # at least this many clean baseline weeks are needed before a change is computed
BASE_FROM, BASE_TO = -35, -8       # daily baseline (BTS): days −35 to −8
MIN_BASE_DAYS = 2                  # each weekday needs at least this many clean baseline days

STEP_H     = 0.25                  # one time slot = 15 minutes
OUT_THRESH = 0.10                  # "substantial outage": at least 10% of customers out (as in the companion outage-duration project)
CALM_HOURS = 24                    # restored = stays below the threshold for this many hours
MIN_PEAK   = 100                   # a county is "affected" only with at least this many customers out at the peak
GAP_FILL_H = 12                    # collection gaps up to this long are filled by interpolation; longer ones stay empty
MIN_DAY_COVER = 0.5                # a day's outage figures need data in at least half of its time slots
MIN_WEEK_DAYS = 4                  # a week's outage figures need data on at least this many days

BAD_DAY_PCT  = 2.0                 # baseline days with a mean outage share of 2% or more are left out of the daily baseline
BAD_WEEK_PCT = 1.0                 # baseline weeks with a mean outage share of 1% or more are left out of the weekly baseline
FLAG_BASE_DAYS = 7                 # flag_base only when more than this many baseline days are left out
FLAG_LATE  = 0.05                  # a day after restoration with a mean outage share above 5% again: a second outage inside the window
PRIOR_DAYS = 70                    # another event within this many days earlier cut this county's power for 24 hours or more: flag_prior
MIN_VISITS = 200                   # fewer weekly visits than this in the baseline on average: sample too small (low_sample)

NET_WEEKS = ["2019-09-09", "2019-09-16", "2019-09-23", "2019-09-30",     # the 8 weeks of the 2019 network (the Monday of each week)
             "2019-10-07", "2019-10-14", "2019-10-21", "2019-10-28"]
BTS_PAGE = 50_000                  # the BTS interface returns at most this many rows per request

## A. Events and study areas

| Event | Day 0 (`t0`, UTC) | Study area (states) |
| --- | --- | --- |
| Isaias 2020 | Landfall at Ocean Isle Beach, NC, 4 August 03:10 | NC, SC, VA, MD, DE, PA, NJ, NY, CT, RI, MA |
| Derecho 2020 | Reaches Cedar Rapids (about 12:30 local), 10 August 17:30 | IA, IL, IN, NE, WI, MO |
| Laura 2020 | Landfall, 27 August 06:00 | LA, TX, AR, MS |
| Delta 2020 | Landfall at Creole, LA, 9 October 23:00 | LA, TX, MS, AR |
| Oklahoma ice storm 2020 | Ice begins to build up around Oklahoma City (about 9 a.m. local), 26 October 14:00 | OK, TX, KS, AR |
| Zeta 2020 | Landfall, 28 October 21:00 | LA, MS, AL, GA, SC, NC, VA |
| Uri 2021 | ERCOT begins rotating outages, 15 February 07:20 | TX, OK, LA, AR, MS |
| Ida 2021 | Landfall, 29 August 16:55 | LA, MS, AL |

- Hurricane times and places come from the official reports of the U.S. National Hurricane Center (NHC); the derecho and the ice storm from the event summaries of the local National Weather Service (NWS) offices; Uri from the FERC and NERC inquiry. The `source` column links to each
- The study areas include the states that were hit and their neighbors, so that there are counties that barely lost power themselves but depend on hard-hit counties
- **Day 0** is the local date of `t0`; **week 0** is the Monday–Sunday week containing day 0. Ida's day 0 is a Sunday, so its effects fall mainly in week 1
- **Events overlap**: Delta hit the same area 43 days after Laura, Zeta came 19 days after Delta, and the ice storm and Delta both reached Texas and Arkansas. Part C marks counties that a previous event had just left without power for 24 hours or more (`flag_prior`)

In [ ]:
# 3. Event table (t0 is in UTC, as in EAGLE-I)
STATE_FIPS = {"AL": "01", "AR": "05", "CT": "09", "DE": "10", "GA": "13", "IL": "17", "IN": "18", "IA": "19",
              "KS": "20", "LA": "22", "MD": "24", "MA": "25", "MS": "28", "MO": "29", "NE": "31", "NJ": "34",
              "NY": "36", "NC": "37", "OK": "40", "PA": "42", "RI": "44", "SC": "45", "TX": "48", "VA": "51", "WI": "55"}
STATES = {v: k for k, v in STATE_FIPS.items()}                     # FIPS → state abbreviation
NHC = "https://www.nhc.noaa.gov/data/tcr/"
FERC = ("https://www.ferc.gov/sites/default/files/2021-09/February%202021%20Cold%20Weather%20Grid%20Operations_"
        "Preliminary%20Findings%20and%20Recommendations-09-23-21_Full%20Presentation.pdf")
CT_, ET_ = "America/Chicago", "America/New_York"
EVENTS = pd.DataFrame([
    dict(key="isaias2020",  name="Isaias",       kind="hurricane",    year=2020, t0="2020-08-04 03:10", place="Ocean Isle Beach, NC", cat="Cat 1",
         states=["NC", "SC", "VA", "MD", "DE", "PA", "NJ", "NY", "CT", "RI", "MA"], tz=ET_, source=NHC + "AL092020_Isaias.pdf"),
    dict(key="derecho2020", name="Derecho",      kind="derecho",      year=2020, t0="2020-08-10 17:30", place="Cedar Rapids, IA", cat="—",
         states=["IA", "IL", "IN", "NE", "WI", "MO"], tz=CT_, source="https://www.weather.gov/dvn/summary_081020"),
    dict(key="laura2020",   name="Laura",        kind="hurricane",    year=2020, t0="2020-08-27 06:00", place="Cameron, LA", cat="Cat 4",
         states=["LA", "TX", "AR", "MS"], tz=CT_, source=NHC + "AL132020_Laura.pdf"),
    dict(key="delta2020",   name="Delta",        kind="hurricane",    year=2020, t0="2020-10-09 23:00", place="Creole, LA", cat="Cat 2",
         states=["LA", "TX", "MS", "AR"], tz=CT_, source=NHC + "AL262020_Delta.pdf"),
    dict(key="okice2020",   name="Oklahoma ice", kind="ice storm",    year=2020, t0="2020-10-26 14:00", place="Oklahoma City area", cat="—",
         states=["OK", "TX", "KS", "AR"], tz=CT_, source="https://www.weather.gov/oun/events-20201026"),
    dict(key="zeta2020",    name="Zeta",         kind="hurricane",    year=2020, t0="2020-10-28 21:00", place="Cocodrie, LA", cat="Cat 3",
         states=["LA", "MS", "AL", "GA", "SC", "NC", "VA"], tz=CT_, source=NHC + "AL282020_Zeta.pdf"),
    dict(key="uri2021",     name="Uri",          kind="winter storm", year=2021, t0="2021-02-15 07:20", place="ERCOT load shed begins", cat="—",
         states=["TX", "OK", "LA", "AR", "MS"], tz=CT_, source=FERC),
    dict(key="ida2021",     name="Ida",          kind="hurricane",    year=2021, t0="2021-08-29 16:55", place="Port Fourchon, LA", cat="Cat 4",
         states=["LA", "MS", "AL"], tz=CT_, source=NHC + "AL092021_Ida.pdf"),
]).set_index("key")
EVENTS["states"] = [[STATE_FIPS[a] for a in st] for st in EVENTS["states"]]
EVENTS["t0"] = pd.to_datetime(EVENTS["t0"])
EVENTS["day0"] = [t.tz_localize("UTC").tz_convert(tz).tz_localize(None).normalize()      # day 0 = the local date
                  for t, tz in zip(EVENTS["t0"], EVENTS["tz"])]
EVENTS["day0"] = EVENTS["day0"].astype("datetime64[ns]")
EVENTS["week0"] = monday(EVENTS["day0"])                                                  # the Monday of week 0
ALL_STATES = sorted({s for st in EVENTS["states"] for s in st})

def event_weeks(key):
    # the event's 12 Mondays
    w0 = EVENTS.loc[key, "week0"]
    return [w0 + pd.Timedelta(weeks=r) for r in range(WEEK_FROM, WEEK_TO + 1)]

def window_days(key):
    # every day of the event window (local dates): exactly these 12 weeks
    w = event_weeks(key)
    return pd.date_range(w[0], w[-1] + pd.Timedelta(days=6), freq="D").astype("datetime64[ns]")

def window_utc(key):
    # start and end of the window in UTC: local midnight of the first day to local midnight after the last day
    days, tz = window_days(key), EVENTS.loc[key, "tz"]
    to_utc = lambda d: d.tz_localize(tz).tz_convert("UTC").tz_localize(None)
    return to_utc(days[0]), to_utc(days[-1] + pd.Timedelta(days=1))

events_out = EVENTS.assign(states=[" ".join(STATES[s] for s in st) for st in EVENTS["states"]])
events_out.to_csv(f"{OUT}/step1b_events.csv")
events_out[["name", "kind", "t0", "day0", "week0", "place", "cat", "states", "source"]]

## B. Counties, population, customers

The list of counties and their population come from Opportunity Insights' `GeoIDs - County.csv` (2019 population). Two reasons for using it: Census population files from 2022 on replace Connecticut's counties with new "planning regions", while EAGLE-I and GeoDS still use the old counties in 2020–2021; and the file comes from the same place as the business data, so the county codes match.

The denominator of the outage share is the same as in the companion outage-duration project: **county customers = county population × statewide customers ÷ state population**. Statewide customers come from EAGLE-I's `coverage_history.csv`, for the event's year. The table below shows electricity customers per person in each state; it should normally lie between 0.4 and 0.6.

In [ ]:
# 4. Download helpers; counties and population; customers per county
def download(url, path, label="", min_bytes=1000):
    # download a file to path; skip it if it already exists
    if os.path.exists(path) and os.path.getsize(path) >= min_bytes:
        return path
    print(f"Downloading {label or url} …")
    with requests.get(url, stream=True, timeout=180, headers={"User-Agent": "Mozilla/5.0"}) as r:
        r.raise_for_status()
        done, shown = 0, 0
        with open(path + ".part", "wb") as f:
            for block in r.iter_content(chunk_size=1 << 22):
                f.write(block); done += len(block)
                if done - shown >= 400e6:
                    shown = done; print(f"  {done / 1e6:,.0f} MB downloaded")
    if done < min_bytes:
        os.remove(path + ".part")
        raise RuntimeError(f"The downloaded file is too small ({done} bytes); the site may have blocked the request. Open {url} in a browser, download it manually, "
                           f"upload it to the outage_mobility folder in Drive under the name {os.path.basename(path)}, then rerun this cell.")
    os.replace(path + ".part", path)
    return path

def get_file(sources, min_bytes=1000):
    # sources is a list of (address, file name). Look for a file of that name in this project's and the companion project's folders first, otherwise download in turn
    for url, name in sources:
        for folder in (OUT, P2):
            p = f"{folder}/{name}"
            if os.path.exists(p) and os.path.getsize(p) >= min_bytes:
                return p
    errors = []
    for url, name in sources:
        try:
            return download(url, f"{OUT}/{name}", name, min_bytes)
        except Exception as err:
            errors.append(f"{name}: {err}")
    raise RuntimeError("\n".join(errors))

OI_URL = "https://raw.githubusercontent.com/OpportunityInsights/EconomicTracker/main/data/"
geo = pd.read_csv(get_file([(OI_URL + quote("GeoIDs - County.csv"), "oi_geoids_county.csv")], 50_000))
geo["fips"] = geo["countyfips"].astype(int).astype(str).str.zfill(5)
geo = geo[geo["fips"].str[:2].isin(ALL_STATES)].set_index("fips").sort_index()
pop = geo["county_pop2019"].astype(float)
county_name = geo["countyname"] + ", " + geo.index.str[:2].map(STATES)
print(f"Counties: {len(geo)} ({len(ALL_STATES)} states), 2019 population {pop.sum() / 1e6:,.1f} million in all")

cov = pd.read_csv(get_file([("https://ndownloader.figshare.com/files/42547714", "coverage_history.csv")]), encoding="utf-8-sig")
cov.columns = [x.strip().lower() for x in cov.columns]
txt = cov["year"].astype(str)                                      # the year is written "1/1/18" or "2018"; both become 2018
yr = pd.to_numeric(txt.str.extract(r"((?:19|20)\d\d)")[0], errors="coerce")
yr = yr.fillna(2000 + pd.to_numeric(txt.str.extract(r"(\d{1,2})\s*$")[0], errors="coerce"))
cov["year"] = yr.astype(int)
cov["state"] = cov["state"].astype(str).str.strip()
cov["total_customers"] = pd.to_numeric(cov["total_customers"].astype(str).str.replace(",", "", regex=False), errors="coerce")
REF_MIN, REF_MAX = int(cov["year"].min()), int(cov["year"].max())

def customers_per_person(state, year):
    # statewide customers ÷ state population in 2019, customers from the nearest year that has data
    ref = min(max(year, REF_MIN), REF_MAX)
    row = cov[(cov["state"] == STATES[state]) & (cov["year"] == ref)]
    if row.empty:
        raise ValueError(f"coverage_history.csv has no record for {STATES[state]} in {ref}")
    return row["total_customers"].iloc[0] / pop[pop.index.str[:2] == state].sum()

den = {}                                                           # den[event] = customers in each county of the study area
for key, ev in EVENTS.iterrows():
    parts = [pop[pop.index.str[:2] == s] * customers_per_person(s, ev["year"]) for s in ev["states"]]
    den[key] = pd.concat(parts).round().sort_index()
    den[key].index.name = "fips"

ratio = pd.DataFrame({y: {STATES[s]: customers_per_person(s, y) for s in ALL_STATES} for y in sorted(set(EVENTS["year"]))})
print("Customers per person in each state:")
display(tidy(ratio, 3).T)
pd.Series({k: len(v) for k, v in den.items()}, name="counties in the study area").to_frame().T

## C. EAGLE-I outages

As in Step 1: EAGLE-I records customers out in each county every 15 minutes (in UTC) and **does not record zeros**. When a state has no record at all at some time, that is treated as a collection gap; gaps of up to 12 hours are filled by interpolation, longer ones stay empty.

Each event is extracted once and saved as `eaglei_<event>.csv.gz`. An existing file is reused if it covers the states and time span this version needs.

Each county gets three kinds of results:

- **Daily figures** (`step1b_outage_daily.csv.gz`): `out_mean_pct` (mean outage share over the day, %), `out_max_pct`, `out_cust_hours` (customer-hours), `out_cover` (share of time slots with data)
- **Weekly figures** (merged into the weekly panel): `out_week_mean_pct` (mean outage share over the week, %) and `out_week_hours` (hours without power per customer during the week)
- **Event labels** (`step1b_outage_event.csv`), defined as in the companion outage-duration project: `peak_pct`, `hours_above`, `days_to_restore`, `affected`, `censored`

Three flags:

| Column | Meaning | What Step 2 does by default |
| --- | --- | --- |
| `base_bad_days` / `flag_base` | Number of baseline days with a mean outage share of 2% or more; flagged only above 7 days | Those days and weeks stay out of the baseline; flagged counties are left out |
| `flag_late` | After restoration, some day had a mean outage share above 5% again (a second outage late in the window) | Only weeks before the second outage are used |
| `flag_prior` / `prior_event` | Another event within the previous 70 days left the county without power for 24 hours or more, so its baseline may still be recovering | Left out, or looked at separately |

In [ ]:
# 5. EAGLE-I: check existing extracts, and download and extract by year what is missing
EAGLEI_FILES = {2020: 42547894, 2021: 42547891}                    # figshare file ids

def parse_time(s):
    # the two timestamp styles that may appear in the files
    for fmt in ("%Y-%m-%d %H:%M:%S", "%m/%d/%y %H:%M", None):
        t = pd.to_datetime(s, format=fmt, errors="coerce")
        if t.notna().mean() > 0.99: break
    return t

def cache_path(key):
    return f"{OUT}/eaglei_{key}.csv.gz"

def cache_ok(key):
    # does an existing file cover the needed states and time span (every state has some outage record over 12 weeks)
    if not os.path.exists(cache_path(key)):
        return False
    c = pd.read_csv(cache_path(key), usecols=["fips", "time"], dtype={"fips": str})
    if c.empty:
        return False
    t, (t0, t1) = pd.to_datetime(c["time"]), window_utc(key)
    return (set(EVENTS.loc[key, "states"]) <= set(c["fips"].str[:2])
            and t.min() <= t0 + pd.Timedelta(days=1) and t.max() >= t1 - pd.Timedelta(days=1))

todo_all = [k for k in EVENTS.index if not cache_ok(k)]
print("To extract:", todo_all or "none")
for year, file_id in EAGLEI_FILES.items():
    todo = [k for k in todo_all if EVENTS.loc[k, "year"] == year]
    if not todo:
        continue
    raw = next((p for p in (f"{OUT}/eaglei_outages_{year}.csv", f"{P2}/eaglei_outages_{year}.csv") if os.path.exists(p)), None)
    if raw is None:                                         # a file downloaded by hand can be placed in the outage_mobility folder
        raw = download(f"https://ndownloader.figshare.com/files/{file_id}", f"{TMP}/eaglei_outages_{year}.csv",
                       f"EAGLE-I {year} (about 1.2 GB)", 300e6)
    header = {x.strip().lower(): x for x in pd.read_csv(raw, nrows=0, encoding="utf-8-sig").columns}
    fips_col, time_col = header["fips_code"], header["run_start_time"]
    out_col = header.get("customers_out", header.get("sum"))
    if out_col is None:
        raise KeyError(f"No customers-out column in the {year} file; the header is: {list(header.values())}")
    states = sorted({s for k in todo for s in EVENTS.loc[k, "states"]})
    parts = {k: [] for k in todo}
    for chunk in pd.read_csv(raw, usecols=[fips_col, out_col, time_col], dtype={fips_col: str},
                             encoding="utf-8-sig", chunksize=2_000_000):
        fips = chunk[fips_col].str.split(".").str[0].str.zfill(5)
        keep = fips.str[:2].isin(states)
        if not keep.any():
            continue
        sub = pd.DataFrame({"fips": fips[keep], "customers_out": pd.to_numeric(chunk.loc[keep, out_col], errors="coerce"),
                            "time": parse_time(chunk.loc[keep, time_col])}).dropna()
        for k in todo:
            t0, t1 = window_utc(k)
            m = sub["fips"].str[:2].isin(EVENTS.loc[k, "states"]) & (sub["time"] >= t0) & (sub["time"] < t1)
            if m.any():
                parts[k].append(sub[m])
    for k in todo:
        one = pd.concat(parts[k]) if parts[k] else pd.DataFrame(columns=["fips", "customers_out", "time"])
        one.to_csv(cache_path(k), index=False)
    if raw.startswith(TMP):
        os.remove(raw)                                      # delete the large raw file once used, to save Colab disk space
    print(f"{year}: extracted {', '.join(todo)}")

In [ ]:
# 6. Each event: 15-minute panel → daily figures, weekly figures, event labels
CALM_SLOTS = int(CALM_HOURS / STEP_H)

def run_lengths(flag):
    # for each True, the length of the run of consecutive Trues it belongs to (0 where False)
    flag = pd.Series(np.asarray(flag, dtype=bool))
    return flag.groupby((~flag).cumsum()).transform("sum").where(flag, 0).values

def first_calm(below, after_peak):
    # the first time after the peak from which the condition holds for the next CALM_HOURS hours; NaT if there is none
    calm = below.astype(float)[::-1].rolling(CALM_SLOTS, min_periods=CALM_SLOTS).min()[::-1].eq(1) & after_peak
    return calm.idxmax().where(calm.any())

def to_long(frame, name):
    # wide table (rows = dates, columns = counties) → long table
    n_rows, n_cols = frame.shape
    return pd.DataFrame({"date": np.repeat(frame.index.values, n_cols), "fips": np.tile(frame.columns.values, n_rows),
                         name: frame.to_numpy().ravel()})

def outage_event(key, e):
    ev, d = EVENTS.loc[key], den[key]
    cols = d.index
    state_of = cols.str[:2]
    t0u, t1u = window_utc(key)
    grid = pd.date_range(t0u, t1u, freq="15min", inclusive="left")
    e = e.assign(slot=e["time"].dt.floor("15min"))
    wide = (e.groupby(["slot", "fips"])["customers_out"].max().unstack("fips")
              .reindex(index=grid, columns=cols).fillna(0.0))

    # whole-state gaps: the state has no record at all in this slot
    gap = pd.DataFrame(False, index=grid, columns=cols)
    for s in ev["states"]:
        has = grid.isin(e.loc[e["fips"].str[:2] == s, "slot"])
        gap.loc[~has, state_of == s] = True
    wide = wide.mask(gap)
    # fill short gaps: interpolate everything, then empty the long gaps again
    limit = int(GAP_FILL_H / STEP_H)
    long_gap = pd.DataFrame(False, index=grid, columns=cols)
    for s in ev["states"]:
        g = gap.loc[:, state_of == s].iloc[:, 0]
        long_gap.loc[run_lengths(g) > limit, state_of == s] = True
    wide = wide.interpolate(method="time", limit_area="inside").mask(long_gap)
    share = wide.div(d, axis=1).clip(upper=1.0)

    # daily figures (calendar days in the event's time zone)
    day = pd.DatetimeIndex(grid.tz_localize("UTC").tz_convert(ev["tz"]).tz_localize(None).normalize()).astype("datetime64[ns]")
    slots = pd.Series(1, index=grid).groupby(day).sum()
    cover = share.notna().groupby(day).sum().div(slots, axis=0)
    ok = cover >= MIN_DAY_COVER
    mean_pct = share.groupby(day).mean().where(ok) * 100
    cust_h = wide.groupby(day).mean().where(ok) * 24
    daily = to_long(mean_pct, "out_mean_pct")
    daily["out_max_pct"] = to_long(share.groupby(day).max().where(ok) * 100, "v")["v"].values
    daily["out_cust_hours"] = to_long(cust_h, "v")["v"].values
    daily["out_cover"] = to_long(cover, "v")["v"].values
    daily.insert(0, "event", key)

    # weekly figures: data needed on at least MIN_WEEK_DAYS days of the week
    wk_of = monday(pd.Series(mean_pct.index)).values
    n_days = mean_pct.notna().groupby(wk_of).sum()
    weekly = pd.DataFrame({
        "out_week_mean_pct": to_long(mean_pct.groupby(wk_of).mean().where(n_days >= MIN_WEEK_DAYS), "v")["v"],
        "out_week_hours": to_long((cust_h.groupby(wk_of).sum() / d).where(n_days >= MIN_WEEK_DAYS), "v")["v"],
    })
    keys = to_long(n_days, "out_week_days")
    weekly = pd.concat([keys.rename(columns={"date": "week"}), weekly], axis=1)
    weekly.insert(0, "event", key)

    # event labels (same definitions as the companion project, counted only from 2 days before day 0)
    t0 = ev["t0"]
    since = grid >= t0 - pd.Timedelta(days=2)
    zone = since & (grid <= t0 + pd.Timedelta(days=7))
    peak_out = wide[zone].max()
    peak_time = wide[zone].fillna(0).idxmax()
    after_peak = pd.DataFrame(grid.values[:, None] >= peak_time.values[None, :], index=grid, columns=cols)
    above = share >= OUT_THRESH
    restored = first_calm(share < OUT_THRESH, after_peak)
    end = grid[-1] + pd.Timedelta(minutes=15)
    until = pd.DataFrame(grid.values[:, None] < restored.fillna(end).values[None, :], index=grid, columns=cols)
    counted = until.copy()
    counted.loc[~since] = False                                    # nothing before 2 days ahead of day 0 is counted
    lab = pd.DataFrame({
        "event": key, "fips": cols, "name": county_name.reindex(cols).values, "state": state_of.map(STATES),
        "customers_est": d.values, "has_records": cols.isin(e["fips"].unique()),
        "peak_customers_out": peak_out.values, "peak_pct": share[zone].max().values * 100,
        "peak_time": peak_time.values, "restored_time": restored.values,
        "hours_above": (above & counted).sum().values * STEP_H,
        "days_to_restore": ((restored - t0) / pd.Timedelta(days=1)).values,
        "customer_hours": wide.where(counted).sum().values * STEP_H,
        "missing_storm_h": (wide.isna() & counted).sum().values * STEP_H,
    })
    lab["affected"] = (lab["peak_pct"] >= OUT_THRESH * 100) & (lab["peak_customers_out"] >= MIN_PEAK)
    lab["censored"] = lab["affected"] & lab["restored_time"].isna()

    # outage days in the baseline; a new outage after restoration
    rel = np.asarray((mean_pct.index - ev["day0"]).days)
    in_base = (rel >= BASE_FROM) & (rel <= BASE_TO)
    lab["base_outage_max_pct"] = mean_pct[in_base].max().values
    lab["base_bad_days"] = (mean_pct[in_base] >= BAD_DAY_PCT).sum().values
    lab["flag_base"] = lab["base_bad_days"] > FLAG_BASE_DAYS
    start = np.where(lab["affected"] & lab["restored_time"].notna(), np.ceil(lab["days_to_restore"]) + 3,
                     np.where(lab["affected"], np.nan, 8.0))         # from the 3rd day after restoration; unaffected counties from day 8
    lab["late_outage_max_pct"] = mean_pct.where(rel[:, None] >= start[None, :]).max().values
    lab["flag_late"] = lab["late_outage_max_pct"] >= FLAG_LATE * 100

    quality = {"filled_gap_h": float((gap & ~long_gap).any(axis=1).sum() * STEP_H),
               "long_gap_h": float(long_gap.any(axis=1).sum() * STEP_H),
               "long_gap_peak_h": float(long_gap[zone].any(axis=1).sum() * STEP_H),
               "eaglei_rows": len(e), "counties_with_records": int(e["fips"].nunique())}
    return wide, daily, weekly, lab, quality

panels, quality = {}, {}
daily_parts, weekly_parts, label_parts = [], [], []
for key in EVENTS.index:
    e = pd.read_csv(cache_path(key), dtype={"fips": str})
    e["time"] = pd.to_datetime(e["time"])
    panels[key], dl, wk, lb, quality[key] = outage_event(key, e)
    daily_parts.append(dl); weekly_parts.append(wk); label_parts.append(lb)
    del e
outage_daily = pd.concat(daily_parts, ignore_index=True)
outage_weekly = pd.concat(weekly_parts, ignore_index=True)
labels = pd.concat(label_parts, ignore_index=True)
outage_daily.to_csv(f"{OUT}/step1b_outage_daily.csv.gz", index=False)
print("All events processed.")

In [ ]:
# 7. Flag for a previous event; summary table
hit24 = labels[labels["affected"] & (labels["hours_above"] >= 24)]
labels["flag_prior"] = False
labels["prior_event"] = ""
for key in EVENTS.index:
    d0 = EVENTS.loc[key, "day0"]
    for k in EVENTS.index:
        gap_days = (d0 - EVENTS.loc[k, "day0"]).days
        if k == key or not (0 < gap_days <= PRIOR_DAYS):
            continue
        m = labels["event"].eq(key) & labels["fips"].isin(hit24.loc[hit24["event"] == k, "fips"])
        labels.loc[m, "flag_prior"] = True
        labels.loc[m, "prior_event"] = (labels.loc[m, "prior_event"] + " " + k).str.strip()
labels.to_csv(f"{OUT}/step1b_outage_event.csv", index=False)

A = labels[labels["affected"]]
summary = pd.DataFrame({
    "counties": labels.groupby("event").size(),
    "with_records": labels.groupby("event")["has_records"].sum(),
    "affected": A.groupby("event").size(),
    "over_24h": A[A["hours_above"] >= 24].groupby("event").size(),
    "over_72h": A[A["hours_above"] >= 72].groupby("event").size(),
    "censored": A.groupby("event")["censored"].sum(),
    "flag_base": A.groupby("event")["flag_base"].sum(),
    "flag_prior": A.groupby("event")["flag_prior"].sum(),
    "flag_late": A.groupby("event")["flag_late"].sum(),
}).reindex(EVENTS.index).fillna(0).astype(int).join(pd.DataFrame(quality).T)
summary

In [ ]:
# 8. Check against the companion project's label table (skipped if the companion project's folder is absent)
p2_labels = next((p for p in (f"{P2}/step2c_labels_county_event.csv", f"{P2}/step2_labels_county_event.csv") if os.path.exists(p)), None)
validation = None
if p2_labels is None:
    print("The companion project's label table was not found; the check is skipped.")
else:
    old = pd.read_csv(p2_labels, dtype={"fips": str})
    old["affected"] = old["affected"].astype(str).str.lower().eq("true")
    m = labels.merge(old[["event", "fips", "hours_above", "peak_pct", "affected"]], on=["event", "fips"], suffixes=("", "_p2"))
    both = m[m["affected"] & m["affected_p2"]]
    rows = []
    for key, g in both.groupby("event"):
        rows.append({"event": key, "counties_both_affected": len(g),
                     "affected_here_only": int((m["event"].eq(key) & m["affected"] & ~m["affected_p2"]).sum()),
                     "affected_p2_only": int((m["event"].eq(key) & ~m["affected"] & m["affected_p2"]).sum()),
                     "spearman_hours_above": spearman(g["hours_above"], g["hours_above_p2"]),
                     "spearman_peak_pct": spearman(g["peak_pct"], g["peak_pct_p2"]),
                     "median_abs_diff_hours": (g["hours_above"] - g["hours_above_p2"]).abs().median()})
    validation = pd.DataFrame(rows).set_index("event")
    validation.to_csv(f"{OUT}/step1b_validation_p2.csv")
    print(f"Compared with: {p2_labels}")
    display(tidy(validation, 3))

## D. BTS travel (secondary measure)

Downloaded the same way as in Step 1. Here it serves only as a comparison: Figure 3 in part H sets it beside GeoDS visits to show why the main measure changed.

The difference from Step 1: baseline days with an outage share of 2% or more stay out of the baseline, and the county is kept.

In [ ]:
# 9. Download BTS daily county travel
BTS_URL  = "https://data.bts.gov/resource/w96p-f2qv.csv"
BTS_COLS = ["date", "county_fips", "pop_stay_at_home", "pop_not_stay_at_home", "trips"]

def bts_get(params):
    # one request; on failure wait and retry, up to 4 times
    for attempt in range(4):
        try:
            r = requests.get(BTS_URL, params=params, timeout=180, headers={"User-Agent": "Mozilla/5.0"})
            r.raise_for_status()
            return pd.read_csv(io.StringIO(r.text), dtype=str)
        except Exception as err:
            if attempt == 3:
                raise RuntimeError(f"The BTS interface failed 4 times in a row: {err}. Rerun this cell in a few minutes; events already downloaded are kept.")
            time.sleep(10 * (attempt + 1))

def bts_state(abbr, first, last):
    # all county records of one state for a range of dates (paging when needed)
    parts, offset = [], 0
    while True:
        part = bts_get({"$select": ",".join(BTS_COLS),
                        "$where": (f"level='County' AND state_code='{abbr}' AND date between "
                                   f"'{first:%Y-%m-%d}T00:00:00' and '{last:%Y-%m-%d}T00:00:00'"),
                        "$order": "county_fips,date", "$limit": BTS_PAGE, "$offset": offset})
        parts.append(part)
        if len(part) < BTS_PAGE:
            break
        offset += BTS_PAGE
    return pd.concat(parts, ignore_index=True)

def read_bts(path):
    m = pd.read_csv(path, dtype={"county_fips": str, "date": str})
    m["fips"] = m["county_fips"].str.split(".").str[0].str.zfill(5)
    m["date"] = as_day(m["date"].str[:10])
    return m

mob = {}
for key in EVENTS.index:
    path, days = f"{OUT}/bts_{key}.csv.gz", window_days(key)
    m = read_bts(path) if os.path.exists(path) else None
    if m is None or not (set(EVENTS.loc[key, "states"]) <= set(m["fips"].str[:2])
                         and m["date"].min() <= days[0] and m["date"].max() >= days[-1]):
        got = pd.concat([bts_state(STATES[s], days[0], days[-1]) for s in EVENTS.loc[key, "states"]], ignore_index=True)
        got.to_csv(path, index=False)
        print(f"{key}: downloaded {len(got):,} rows")
        m = read_bts(path)
    for col in ["pop_stay_at_home", "pop_not_stay_at_home", "trips"]:
        m[col] = pd.to_numeric(m[col], errors="coerce")
    m["share_not_home"] = m["pop_not_stay_at_home"] / (m["pop_stay_at_home"] + m["pop_not_stay_at_home"])
    m = m[m["date"].isin(days)]
    mob[key] = m[["fips", "date", "trips", "share_not_home"]].drop_duplicates(["fips", "date"])

pd.DataFrame({k: {"rows": len(m), "counties": m["fips"].nunique(), "first": m["date"].min(), "last": m["date"].max()}
              for k, m in mob.items()}).T

## E. GeoDS visits (main measure)

The GeoDS Lab (University of Wisconsin–Madison, Song Gao's group) computed weekly county-to-county visits from SafeGraph phone data: visitors who live in county A and visit shops and places in county B (`visitor_flows`). This cell downloads the 48 weeks the events need plus the 8 weeks of 2019 used for the network, keeps only rows that start or end inside a study area, and saves each week to the `geods_weeks` folder, together with the national total of visits that week.

Each county gets four numbers per week:

| Column | Meaning | What it reflects |
| --- | --- | --- |
| `visits_in` | Visits to places in the county, wherever the visitors live | **Local activity** (main outcome) |
| `visits_in_nonlocal` | Of these, visits from other counties | Outside visitors |
| `visits_res` | Visits made by the county's residents, anywhere | Residents' activity |
| `visits_res_local` | Of these, visits inside the home county | Residents' local activity |

SafeGraph's sample changes from week to week, so each number is first divided by the national total of visits that week (unit: per million visits nationwide) and only then compared with the baseline.

Two more things are saved for Step 3: each event's county-pair flows over its 12 weeks for pairs that touch the study area (`step1b_edges_<event>.csv.gz`), and the 2019 dependency network (`step1b_network_2019.csv.gz`). Tropical Storm Imelda (Houston, 17–19 September) falls inside the 8 weeks of 2019; summed over 8 weeks its effect is small.

In [ ]:
# 10. Download the GeoDS weekly files, filter them and save them to geods_weeks
GEODS_URL = ("https://raw.githubusercontent.com/GeoDS/COVID19USFlows-WeeklyFlows/master/weekly_flows/"
             "county2county/weekly_county2county_{}.csv")
TOTALS = f"{GEO_CACHE}/week_totals.csv"
net_weeks = [pd.Timestamp(w) for w in NET_WEEKS]
need = sorted({w for k in EVENTS.index for w in event_weeks(k)} | set(net_weeks))

def week_path(w):
    return f"{GEO_CACHE}/geods_{w:%Y_%m_%d}.csv.gz"

totals = pd.read_csv(TOTALS, parse_dates=["week"]) if os.path.exists(TOTALS) else pd.DataFrame(columns=["week", "total_visits"])
totals["week"] = as_day(totals["week"])
done = set(totals["week"])
for i, w in enumerate(need, 1):
    if w in done and os.path.exists(week_path(w)):
        continue
    tag = f"{w:%Y_%m_%d}"
    p = download(GEODS_URL.format(tag), f"{TMP}/geods_{tag}.csv", f"GeoDS {w:%Y-%m-%d} ({i}/{len(need)})", 10e6)
    f = pd.read_csv(p, usecols=["geoid_o", "geoid_d", "visitor_flows"], dtype={"geoid_o": str, "geoid_d": str})
    f["geoid_o"], f["geoid_d"] = f["geoid_o"].str.zfill(5), f["geoid_d"].str.zfill(5)
    keep = f["geoid_o"].str[:2].isin(ALL_STATES) | f["geoid_d"].str[:2].isin(ALL_STATES)
    f[keep].to_csv(week_path(w), index=False, compression={"method": "gzip", "compresslevel": 1})
    totals = pd.concat([totals[totals["week"] != w], pd.DataFrame({"week": [w], "total_visits": [f["visitor_flows"].sum()]})])
    totals["week"] = as_day(totals["week"])
    totals = totals.sort_values("week").reset_index(drop=True)
    totals.to_csv(TOTALS, index=False)                        # the week's file is saved before its total, so an interrupted run is completed on a rerun
    os.remove(p)
totals["total_visits"] = pd.to_numeric(totals["total_visits"])
missing = [w for w in need if not (w in set(totals["week"]) and os.path.exists(week_path(w)))]
print(f"GeoDS: {len(need)} weeks needed, {len(missing)} missing")

In [ ]:
# 11. Four visit counts per county and week; county-pair flows per event; the 2019 network
VISITS = ["visits_in", "visits_in_nonlocal", "visits_res", "visits_res_local"]
REGION = sorted(set().union(*[set(v.index) for v in den.values()]))
weeks_of = {k: set(event_weeks(k)) for k in EVENTS.index}
county_rows, net_parts = [], []
edges = {k: [] for k in EVENTS.index}
for w in need:
    f = pd.read_csv(week_path(w), dtype={"geoid_o": str, "geoid_d": str})
    if w in net_weeks:
        net_parts.append(f)
    users = [k for k in EVENTS.index if w in weeks_of[k]]
    if not users:
        continue
    local = f["geoid_o"] == f["geoid_d"]
    agg = pd.DataFrame({
        "visits_in": f.groupby("geoid_d")["visitor_flows"].sum(),
        "visits_in_nonlocal": f[~local].groupby("geoid_d")["visitor_flows"].sum(),
        "visits_res": f.groupby("geoid_o")["visitor_flows"].sum(),
        "visits_res_local": f[local].groupby("geoid_o")["visitor_flows"].sum(),
    }).reindex(REGION).fillna(0)
    county_rows.append(agg.rename_axis("fips").reset_index().assign(week=w))
    for k in users:
        st = EVENTS.loc[k, "states"]
        m = f["geoid_o"].str[:2].isin(st) | f["geoid_d"].str[:2].isin(st)
        edges[k].append(f[m].assign(week=w))
geo_cw = pd.concat(county_rows, ignore_index=True)
geo_cw["week"] = as_day(geo_cw["week"])
geo_cw = geo_cw.merge(totals, on="week", how="left")

for k, parts in edges.items():
    ed = pd.concat(parts, ignore_index=True)
    ed["week"] = as_day(ed["week"])
    ed["rel_week"] = (ed["week"] - EVENTS.loc[k, "week0"]).dt.days // 7
    ed[["week", "rel_week", "geoid_o", "geoid_d", "visitor_flows"]].to_csv(f"{OUT}/step1b_edges_{k}.csv.gz", index=False,
                                                                           compression={"method": "gzip", "compresslevel": 1})
    print(f"{k}: {len(ed):,} county-pair rows")

net = (pd.concat(net_parts, ignore_index=True).groupby(["geoid_o", "geoid_d"], as_index=False)
         .agg(visitor_flows=("visitor_flows", "sum"), weeks=("visitor_flows", "size")))
net.to_csv(f"{OUT}/step1b_network_2019.csv.gz", index=False)
out_tot = net.groupby("geoid_o")["visitor_flows"].sum()
stay = (net[net["geoid_o"] == net["geoid_d"]].set_index("geoid_o")["visitor_flows"] / out_tot).reindex(REGION)
print(f"2019 network: {len(net):,} county pairs; of the {len(REGION)} study-area counties, {out_tot.index.isin(REGION).sum()} appear as an origin; "
      f"share of visits that stay in the home county (median) {stay.median():.2f}")

## F. Business and spending (Opportunity Insights)

The same files as in Step 1, made weekly here:

- **Womply** (small-business revenue `revenue`, small businesses open `merchants`): already weekly, with weeks ending on Sunday, which matches GeoDS's Monday–Sunday weeks exactly. Placed by **the county where the business is**
- **Affinity** (card spending `spend`): each daily value is the average of the 7 days up to that day, so the Sunday value stands for the week. Placed by **the county where the cardholder lives**

Values are "change since January 2020, seasonally adjusted"; part G subtracts the baseline, giving the change from the baseline in percentage points (`*_chg`). The county files only have totals, without breakdowns by income.

In [ ]:
# 12. Weekly Womply and Affinity data
aff = pd.read_csv(get_file([(OI_URL + quote("Affinity - County - Daily.csv"), "oi_affinity_county.csv")], 1_000_000), dtype={"spend_all": str})
wom = pd.read_csv(get_file([(OI_URL + quote("Womply - County - Weekly.csv"), "oi_womply_county.csv")], 100_000))
aff["date"] = as_day(pd.to_datetime(aff[["year", "month", "day"]]))
wom["date"] = as_day(pd.to_datetime(wom[["year", "month", "day_endofweek"]].rename(columns={"day_endofweek": "day"})))
for d in (aff, wom):
    d["fips"] = d["countyfips"].astype(int).astype(str).str.zfill(5)
aff = aff[aff["date"].dt.dayofweek.eq(6)]                          # Sundays only
biz_w = pd.concat([
    pd.DataFrame({"fips": aff["fips"], "week": aff["date"] - pd.Timedelta(days=6), "series": "spend",
                  "value": pd.to_numeric(aff["spend_all"], errors="coerce")}),            # "." marks a masked value
    pd.DataFrame({"fips": wom["fips"], "week": wom["date"] - pd.Timedelta(days=6), "series": "revenue", "value": wom["revenue_all"]}),
    pd.DataFrame({"fips": wom["fips"], "week": wom["date"] - pd.Timedelta(days=6), "series": "merchants", "value": wom["merchants_all"]}),
], ignore_index=True).dropna(subset=["value"])
biz_w = biz_w[biz_w["fips"].isin(REGION)].pivot_table(index=["fips", "week"], columns="series", values="value").reset_index()
biz_w.columns.name = None
print("Counties with data:", {s: int(biz_w.dropna(subset=[s])["fips"].nunique()) for s in ["revenue", "merchants", "spend"]})

## G. Merge into the panels

**Weekly panel** (`step1b_panel_weekly.csv.gz`, the main result): each row is "event × county × week".

| Column | Meaning |
| --- | --- |
| `rel_week` | Week number (week 0 contains day 0) |
| `out_week_mean_pct`, `out_week_hours` | The week's outage figures |
| `visits_in` etc. | Raw visit counts; `*_pm` is per million visits nationwide |
| `visits_in_chg` etc. | Change from the baseline, in **log points** (100 × difference in logs, close to a percentage when small) |
| `revenue_chg`, `merchants_chg`, `spend_chg` | Change from the baseline (percentage points of the January 2020 level) |
| `base_weeks` | How many weeks the visits baseline uses |
| `low_sample` | Fewer than 200 visits per week on average in the baseline; changes are very unstable |

Baseline: the weeks among weeks −5 to −2 in which the county's mean outage share stayed below 1%. At least 2 such weeks are needed; otherwise the change is left blank.

**Daily panel** (`step1b_panel_daily.csv.gz`): daily outage figures and BTS travel, laid out as in Step 1; the baseline leaves out days with an outage share of 2% or more.

In [ ]:
# 13. Daily panel (outages and BTS)
frames = []
for key in EVENTS.index:
    g = pd.MultiIndex.from_product([den[key].index, window_days(key)], names=["fips", "date"]).to_frame(index=False)
    g.insert(0, "event", key)
    frames.append(g)
daily = pd.concat(frames, ignore_index=True)
daily["date"] = as_day(daily["date"])
daily["rel_day"] = (daily["date"] - daily["event"].map(EVENTS["day0"])).dt.days
daily["weekday"] = daily["date"].dt.dayofweek
daily = daily.merge(outage_daily.assign(date=as_day(outage_daily["date"])), on=["event", "fips", "date"], how="left")
daily = daily.merge(pd.concat([m.assign(event=k) for k, m in mob.items()], ignore_index=True), on=["event", "fips", "date"], how="left")

clean = daily["rel_day"].between(BASE_FROM, BASE_TO) & ~(daily["out_mean_pct"] >= BAD_DAY_PCT)
base = (daily[clean].groupby(["event", "fips", "weekday"])
          .agg(base_trips=("trips", "mean"), base_not_home=("share_not_home", "mean"), base_n=("trips", "count"))
          .reset_index())
daily = daily.merge(base, on=["event", "fips", "weekday"], how="left")
ok = (daily["base_n"] >= MIN_BASE_DAYS) & (daily["base_trips"] > 0)
daily["trips_change_pct"] = ((daily["trips"] / daily["base_trips"] - 1) * 100).where(ok)
daily["not_home_change_pp"] = ((daily["share_not_home"] - daily["base_not_home"]) * 100).where(ok)
daily.to_csv(f"{OUT}/step1b_panel_daily.csv.gz", index=False)
print(f"Daily panel: {len(daily):,} rows")

In [ ]:
# 14. Weekly panel (outages, GeoDS visits, business and spending)
frames = []
for key in EVENTS.index:
    g = pd.MultiIndex.from_product([den[key].index, event_weeks(key)], names=["fips", "week"]).to_frame(index=False)
    g.insert(0, "event", key)
    frames.append(g)
W = pd.concat(frames, ignore_index=True)
W["week"] = as_day(W["week"])
W["state"] = W["fips"].str[:2].map(STATES)
W["rel_week"] = (W["week"] - W["event"].map(EVENTS["week0"])).dt.days // 7
W = W.merge(outage_weekly.assign(week=as_day(outage_weekly["week"])), on=["event", "fips", "week"], how="left")
W = W.merge(geo_cw.assign(week=as_day(geo_cw["week"])), on=["fips", "week"], how="left")
W = W.merge(biz_w.assign(week=as_day(biz_w["week"])), on=["fips", "week"], how="left")
for c in VISITS:
    W[c + "_pm"] = W[c] / W["total_visits"] * 1e6

clean = W["rel_week"].between(*BASE_WEEKS) & ~(W["out_week_mean_pct"] >= BAD_WEEK_PCT)
by = [W["event"], W["fips"]]
def change(v):
    # change from the baseline: v minus its mean over clean baseline weeks; blank with fewer than MIN_BASE_WEEKS clean weeks
    b = v.where(clean).groupby(by).transform("mean")
    n = v.where(clean).groupby(by).transform("count")
    return ((v - b) * 100).where(n >= MIN_BASE_WEEKS), n
for c in VISITS:
    W[c + "_chg"], n = change(np.log(W[c + "_pm"].where(W[c + "_pm"] > 0)))
    if c == "visits_in":
        W["base_weeks"] = n
for c in ["revenue", "merchants", "spend"]:
    W[c + "_chg"], _ = change(W[c])
W["base_visits_in"] = W["visits_in"].where(clean).groupby(by).transform("mean")
W["low_sample"] = ~(W["base_visits_in"] >= MIN_VISITS)
W.to_csv(f"{OUT}/step1b_panel_weekly.csv.gz", index=False)
print(f"Weekly panel: {len(W):,} rows, {W['fips'].nunique()} counties, {W['event'].nunique()} events")
tidy(W.groupby("event")[["out_week_mean_pct", "visits_in_chg", "revenue_chg", "spend_chg"]]
      .agg(lambda x: x.notna().mean() * 100).reindex(EVENTS.index).rename(columns=lambda c: c + "_filled_pct"))

## H. Coverage, three figures, checklist

The counties of each event are put into four groups by outage duration: no substantial outage (peak below 10%), under 24 hours, 24–72 hours, and 72 hours or more.

**All figures in this version show differences from counties without a substantial outage**: each week, the median of the gray group (counties without a substantial outage) is subtracted from every group. In Step 1's Figure 1 all groups moved up and down together because of holidays and seasons; only after taking that out does the effect of outages show. Counties with `flag_base`, `flag_prior` or `low_sample` are left out of the figures.

In [ ]:
# 15. Outage groups and the coverage table
labels["group"] = np.select([~labels["affected"], labels["hours_above"] < 24, labels["hours_above"] < 72],
                            ["none", "lt24", "24to72"], "ge72")
labels.to_csv(f"{OUT}/step1b_outage_event.csv", index=False)
W = W.merge(labels[["event", "fips", "group", "flag_base", "flag_prior", "flag_late", "hours_above"]], on=["event", "fips"], how="left")
use_w = W[~W["flag_base"] & ~W["flag_prior"] & ~W["low_sample"]]
net_origins = set(net["geoid_o"])

rows = []
for key in EVENTS.index:
    l = labels[labels["event"] == key]
    a24 = l[l["affected"] & (l["hours_above"] >= 24)]
    w01 = W[(W["event"] == key) & W["rel_week"].isin([0, 1])]
    def share_with(col, sub=a24):
        ok = w01.groupby("fips")[col].apply(lambda x: x.notna().all())
        return ok.reindex(sub["fips"], fill_value=False).mean() if len(sub) else np.nan
    def n_with(col, sub=a24):
        ok = w01.groupby("fips")[col].apply(lambda x: x.notna().all())
        return int(ok.reindex(sub["fips"], fill_value=False).sum())
    low = W[(W["event"] == key) & (W["rel_week"] == 0)].set_index("fips")["low_sample"]
    rows.append({"event": key, "counties": len(l), "affected": int(l["affected"].sum()),
                 "over_24h": len(a24), "over_72h": int((l["group"] == "ge72").sum()),
                 "visits_share_over_24h": share_with("visits_in_chg"),
                 "low_sample_over_24h": int(low.reindex(a24["fips"], fill_value=True).sum()),
                 "flag_base_affected": int((l["affected"] & l["flag_base"]).sum()),
                 "flag_base_share_affected": l.loc[l["affected"], "flag_base"].mean() if l["affected"].any() else np.nan,
                 "flag_prior_affected": int((l["affected"] & l["flag_prior"]).sum()),
                 "flag_late_affected": int((l["affected"] & l["flag_late"]).sum()),
                 "revenue_over_24h": n_with("revenue_chg"), "spend_over_24h": n_with("spend_chg"),
                 "network_share": l["fips"].isin(net_origins).mean(),
                 "long_gap_peak_h": quality[key]["long_gap_peak_h"]})
coverage = pd.DataFrame(rows).set_index("event")
coverage.to_csv(f"{OUT}/step1b_coverage.csv")
tidy(coverage, 2)

In [ ]:
# 16. Figure 1: visits to places in the county, each group minus the median of the "no substantial outage" group, by week
MIN_GROUP = 3                                                                  # groups with fewer than 3 counties are not drawn
def relative(df, col, time_col):
    # for each event and time point, subtract the median of the "no substantial outage" group
    ref = df[df["group"] == "none"].groupby(["event", time_col])[col].median().rename("ref")
    out = df.merge(ref.reset_index(), on=["event", time_col], how="left")
    return out[col] - out["ref"], out

fig, axes = plt.subplots(2, 4, figsize=(15, 7.8), sharey=True)
v, U = relative(use_w, "visits_in_chg", "rel_week")
U = U.assign(rel=v.values)
counts1 = {}
for ax, key in zip(axes.flat, EVENTS.index):
    sub = U[U["event"] == key]
    ax.axvline(0, color=FAINT, lw=1); ax.axhline(0, color=FAINT, lw=1)
    for g, label, color, ls in GROUPS:
        s = sub[sub["group"] == g].dropna(subset=["rel"])
        n = s["fips"].nunique()
        if n < MIN_GROUP:
            continue
        med = s.groupby("rel_week")["rel"].median()
        ax.plot(med.index, med.values, color=color, lw=2, ls=ls, marker="o", ms=4)
        counts1[(key, label)] = n
    ev = EVENTS.loc[key]
    ax.set_title(f"{ev['name']} {ev['year']}")
    ax.set_xticks(WEEK_TICKS)
for ax in axes[-1]:
    ax.set_xlabel("Week (week 0 contains day 0)")
for ax in axes[:, 0]:
    ax.set_ylabel("Visits to places in the county\n(log points, relative to no-outage counties)")
handles = [plt.Line2D([], [], color=color, lw=2, ls=ls, marker="o", ms=4) for _, _, color, ls in GROUPS]
fig.legend(handles, [label for _, label, _, _ in GROUPS], title="Outage duration", loc="upper left",
           bbox_to_anchor=(0.01, 0.95), ncol=len(GROUPS), fontsize=9, title_fontsize=9, alignment="left")
fig.suptitle("Visits to places in the county against the baseline, median county by outage duration, minus the no-outage median",
             x=0.01, ha="left", fontweight="bold", fontsize=12)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(f"{OUT}/step1b_fig1_visits_by_outage.png"); plt.show()

# how many counties each line of Figure 1 uses
pd.Series(counts1, dtype=int).rename_axis(["event", "group"]).unstack("group").reindex(
    index=EVENTS.index, columns=[label for _, label, _, _ in GROUPS]).fillna(0).astype(int)

In [ ]:
# 17. Figure 2: business revenue (where businesses are) and card spending (where people live), minus the no-outage median
rows_ = [("revenue_chg", "Small-business revenue"), ("spend_chg", "Card spending")]
fig, axes = plt.subplots(2, len(EVENTS), figsize=(20, 6.8), sharey="row", sharex=True)
counts = {}
for i, (col, row_label) in enumerate(rows_):
    v, U = relative(use_w, col, "rel_week")
    U = U.assign(rel=v.values)
    for j, key in enumerate(EVENTS.index):
        ax = axes[i, j]
        sub = U[U["event"] == key]
        ax.axvline(0, color=FAINT, lw=1); ax.axhline(0, color=FAINT, lw=1)
        drawn = 0
        for g, label, color, ls in GROUPS:
            s = sub[sub["group"] == g].dropna(subset=["rel"])
            n = s["fips"].nunique()
            if n < MIN_GROUP:
                continue
            med = s.groupby("rel_week")["rel"].median()
            ax.plot(med.index, med.values, color=color, lw=2, ls=ls, marker="o", ms=3.5)
            counts[(key, col, label)] = n
            drawn += 1
        if drawn == 0:
            ax.text(0.5, 0.5, "No data", transform=ax.transAxes, ha="center", va="center", color=MUTED)
        if i == 0:
            ev = EVENTS.loc[key]; ax.set_title(f"{ev['name']} {ev['year']}")
        if j == 0:
            ax.set_ylabel(f"{row_label}\n(pp of Jan 2020, relative\nto no-outage counties)")
        if i == 1:
            ax.set_xlabel("Week")
            ax.set_xticks(WEEK_TICKS)
handles = [plt.Line2D([], [], color=color, lw=2, ls=ls) for _, _, color, ls in GROUPS]
fig.legend(handles, [label for _, label, _, _ in GROUPS], title="Outage duration", loc="upper left",
           bbox_to_anchor=(0.01, 0.95), ncol=len(GROUPS), fontsize=9, title_fontsize=9, alignment="left")
fig.suptitle("Business revenue and card spending against the baseline, median county by outage duration, minus the no-outage median",
             x=0.01, ha="left", fontweight="bold", fontsize=12)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(f"{OUT}/step1b_fig2_business_by_outage.png"); plt.show()

# how many counties each line of Figure 2 uses
pd.Series(counts, dtype=int).rename_axis(["event", "series", "group"]).unstack("group").reindex(
    columns=[label for _, label, _, _ in GROUPS]).fillna(0).astype(int)

In [ ]:
# 18. Figure 3: how GeoDS visits and BTS trips respond to outages, same counties, same two weeks
v, U = relative(use_w, "visits_in_chg", "rel_week")
U = U.assign(rel=v.values)
geo01 = U[U["rel_week"].isin([0, 1])].groupby(["event", "fips"]).agg(visits=("rel", "mean"), hours=("out_week_hours", "sum"))
D = daily.merge(labels[["event", "fips", "group", "flag_base", "flag_prior"]], on=["event", "fips"])
D["rel_week"] = (monday(D["date"]) - D["event"].map(EVENTS["week0"])).dt.days // 7
D = D[~D["flag_base"] & ~D["flag_prior"] & D["rel_week"].isin([0, 1])]
v, D = relative(D, "trips_change_pct", "date")
D = D.assign(rel=v.values)
bts01 = D.groupby(["event", "fips"])["rel"].mean().rename("trips")
cmp_ = geo01.join(bts01, how="inner").reset_index().merge(labels[["event", "fips", "group"]], on=["event", "fips"])
cmp_ = cmp_[cmp_["group"].isin(["24to72", "ge72"]) & (cmp_["hours"] > 0)]
cmp_.to_csv(f"{OUT}/step1b_measure_comparison.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharex=True)
for ax, col, ylab in [(axes[0], "visits", "GeoDS: visits to places in the county\n(log points, relative to no-outage counties)"),
                      (axes[1], "trips", "BTS: trips by residents\n(%, relative to no-outage counties)")]:
    ok = cmp_.dropna(subset=[col])
    ax.axhline(0, color=FAINT, lw=1)
    ax.scatter(ok["hours"], ok[col], s=16, color=C_POINT, edgecolor=SURFACE, linewidth=0.5, alpha=0.7)
    ax.set_xscale("log")
    ax.xaxis.set_major_locator(LogLocator(base=10, subs=(1.0, 2.0, 5.0)))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:g}"))
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.set_xlabel("Hours without power per customer, weeks 0 and 1")
    ax.set_ylabel(ylab)
    ax.set_title(f"Spearman {spearman(ok['hours'], ok[col]):.2f} (n = {len(ok)} county-events)", fontsize=10, color=MUTED)
fig.suptitle("Counties with outages of 24 hours or more: mean change over weeks 0 and 1, all events",
             x=0.01, ha="left", fontweight="bold", fontsize=12)
fig.tight_layout()
fig.savefig(f"{OUT}/step1b_fig3_visits_vs_trips.png"); plt.show()

**Reading the three figures**

- **Figure 1**: the gray dashed line is 0 (it is the reference group); the table under the figure gives the number of counties behind each line. If outages reduce visits, the longer a group's outage, the deeper its drop in weeks 0 and 1 and the slower its return. Ida's day 0 is a Sunday, so its drop should come in week 1
- **Figure 2**: the top row is revenue where the businesses are, the bottom row spending where people live. Step 1 found that residents of hard-hit counties sometimes spend more (evacuation, rebuilding) while local business revenue falls, so spending may move to other counties. Step 3 tests this
- **Figure 3**: the same counties with outages of 24 hours or more, over the same two weeks. Left, GeoDS visits; right, BTS trips. If the left panel falls with outage duration and the right one does not, the change of measure is justified
- All three figures are descriptive. "Relative to counties without a substantial outage" removes only shared ups and downs; the effect of wind and rain themselves is still in there, and Step 2 controls for wind speed and rainfall

In [ ]:
# 19. Step 1b checklist
def verdict(passed):
    if passed == "info":
        return "for information"
    return "pending" if passed is None else ("pass" if passed else "fail")

n_gap_ok = int((coverage["long_gap_peak_h"] == 0).sum())
geo_min  = float(coverage["visits_share_over_24h"].min())
val_min  = None if validation is None or validation.empty else float(validation["spearman_hours_above"].min())
net_min  = float(coverage["network_share"].min())
base_max = float(coverage["flag_base_share_affected"].max())
prior_n  = int(coverage["flag_prior_affected"].sum())
rev_n    = int(coverage["revenue_over_24h"].sum())

checks = [
    ("Events without unfilled long gaps around the peak", f"{n_gap_ok} / {len(EVENTS)}", f"= {len(EVENTS)}", n_gap_ok == len(EVENTS)),
    ("All GeoDS weeks downloaded", f"{len(need) - len(missing)} / {len(need)}", f"= {len(need)}", len(missing) == 0),
    ("Share of counties with 24+ h outages that have a visits change in weeks 0 and 1, lowest event", round(geo_min, 2), "≥ 0.9", geo_min >= 0.9),
    ("Agreement with the companion project's outage-duration ranking (Spearman), lowest event", "—" if val_min is None else round(val_min, 2), "≥ 0.9", None if val_min is None else val_min >= 0.9),
    ("Share of study-area counties present in the 2019 network, lowest event", round(net_min, 3), "≥ 0.95", net_min >= 0.95),
    ("Share of affected counties with flag_base, highest event", round(base_max, 2), "≤ 0.1", base_max <= 0.1),
    ("Affected counties with flag_prior (eight events together)", prior_n, "—", "info"),
    ("Counties with Womply data and 24+ h outages (eight events together)", rev_n, "≥ 30", rev_n >= 30),
]
check_tab = pd.DataFrame(checks, columns=["Indicator", "Value", "Threshold", "passed"])
check_tab["Verdict"] = check_tab.pop("passed").map(verdict)
check_tab.to_csv(f"{OUT}/step1b_checklist.csv", index=False)
check_tab